# Gamma calibration: how many communities should we expect?

Before committing to a (gamma, omega) grid for the full parameter sweep, this notebook answers one question empirically: **which gamma values produce roughly 5–7 communities** on a real scan, at the fixed `omega = 0.25` we're using as our interslice-coupling default?

It reuses the same scan found in:

`outputs/window_connectivity/Bf_DTA_anes/FD_cut30frames_bandpass0.01-0.1_cut30edges_mot6_CSF_smooth6/sub-ag230908/sub-ag230908a`

For each gamma candidate, it runs temporal Leiden **5 times** (not 20 — this is a quick calibration probe, not a precision estimate) and reports the per-window community-count distribution (min/median/mean/max). CPM's resolution behavior is graph-specific and often steps rather than scales smoothly with gamma, so this is read off real data rather than assumed.

In [3]:
import importlib.util
import multiprocessing
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import leidenalg as la
import numpy as np
import pandas as pd

## Configuration

All available windows from the first scan in the requested mouse folder are used. `OMEGA` is fixed at our current default; `GAMMA_CANDIDATES` is the set of values to probe.

In [4]:
WORKING_DIR = Path.cwd().resolve()
ROOT = WORKING_DIR.parent if WORKING_DIR.name == "scripts" else WORKING_DIR

WINDOW_DIR = (
    ROOT
    / "outputs"
    / "window_connectivity"
    / "Bf_DTA_anes"
    / "FD_cut30frames_bandpass0.01-0.1_cut30edges_mot6_CSF_smooth6"
    / "sub-ag230908"
    / "sub-ag230908a"
)

GAMMA_CANDIDATES = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
OMEGA = 0.25
N_RUNS = 5
RANDOM_SEED = 20260615

OUTPUT_DIR = ROOT / "outputs" / "community_interpretation" / "gamma_calibration_omega-0.25"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert WINDOW_DIR.is_dir(), f"Missing input directory: {WINDOW_DIR}"
WINDOW_DIR

PosixPath('/home/luca.galli-1/neuro3ducate/mouse-fmri-dyncomm/outputs/window_connectivity/Bf_DTA_anes/FD_cut30frames_bandpass0.01-0.1_cut30edges_mot6_CSF_smooth6/sub-ag230908/sub-ag230908a')

## Select the first scan

Files are sorted by window number. The scan prefix of the first file determines the scan analyzed below.

In [5]:
all_window_paths = sorted(WINDOW_DIR.glob("*_window_*.csv"))
assert all_window_paths, f"No connectivity windows found in {WINDOW_DIR}"

first_stem = all_window_paths[0].stem
SCAN_ID = first_stem.rsplit("_window_", 1)[0]
window_paths = [
    path for path in all_window_paths
    if path.stem.rsplit("_window_", 1)[0] == SCAN_ID
]

print(f"Scan: {SCAN_ID}")
print(f"Windows: {len(window_paths)}")
print(f"First window: {window_paths[0].name}")
print(f"Last window:  {window_paths[-1].name}")

Scan: sub-ag230908a_EXP_bold_parcellated
Windows: 650
First window: sub-ag230908a_EXP_bold_parcellated_window_0008.csv
Last window:  sub-ag230908a_EXP_bold_parcellated_window_0671.csv


## Load the temporal connectivity graphs

The notebook reuses the graph construction from `04_run_community_detection.py`: one vertex per ROI, positive Pearson correlations as weighted edges, and a zero diagonal.

In [6]:
scripts_dir = ROOT / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

spec = importlib.util.spec_from_file_location(
    "community_detection_04",
    scripts_dir / "04_run_community_detection.py",
)
cd04 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = cd04
spec.loader.exec_module(cd04)

graphs = []
roi_names = None
window_ids = []

for path in window_paths:
    frame = cd04.read_window(str(path))
    current_rois = list(frame.columns)
    if roi_names is None:
        roi_names = current_rois
    else:
        assert current_rois == roi_names, f"ROI order changed in {path}"
    graphs.append(cd04.build_igraph(frame))
    window_ids.append(int(path.stem.rsplit("_window_", 1)[1]))

print(f"Loaded {len(graphs)} graphs with {len(roi_names)} ROIs each")
print("ROI order:", ", ".join(roi_names))

Loaded 650 graphs with 16 ROIs each
ROI order: DMNa, DMNp, SAL, OLF, STR, AUD, VIS, TH, MOp, SSp, SSs, HCa, SUB, CTXsp, BF, HY


## Run temporal Leiden 5 times per gamma candidate

For each gamma in `GAMMA_CANDIDATES`, 5 independent Leiden runs are launched (omega fixed). All `len(GAMMA_CANDIDATES) * N_RUNS` runs are parallelized across worker processes. The saved random seeds allow the result to be reproduced exactly.

In [7]:
_WORKER_GRAPHS = None


def _initialize_worker(worker_graphs):
    global _WORKER_GRAPHS
    _WORKER_GRAPHS = worker_graphs


def _run_single_leiden(gamma, seed):
    memberships, improvement = la.find_partition_temporal(
        _WORKER_GRAPHS,
        la.CPMVertexPartition,
        interslice_weight=OMEGA,
        vertex_id_attr="id",
        weight_attr="weight",
        resolution_parameter=gamma,
        seed=int(seed),
    )
    return gamma, np.asarray(memberships, dtype=np.int32), float(improvement)

In [8]:
seed_generator = np.random.default_rng(RANDOM_SEED)
tasks = [
    (gamma, int(seed))
    for gamma in GAMMA_CANDIDATES
    for seed in seed_generator.integers(0, 2**31 - 1, size=N_RUNS, dtype=np.int64)
]

memberships_by_gamma = {gamma: [] for gamma in GAMMA_CANDIDATES}
improvements_by_gamma = {gamma: [] for gamma in GAMMA_CANDIDATES}

run_workers = min(len(tasks), multiprocessing.cpu_count())
mp_context = multiprocessing.get_context("fork")

with ProcessPoolExecutor(
    max_workers=run_workers,
    mp_context=mp_context,
    initializer=_initialize_worker,
    initargs=(graphs,),
) as executor:
    futures = [executor.submit(_run_single_leiden, gamma, seed) for gamma, seed in tasks]
    for completed, future in enumerate(as_completed(futures), start=1):
        gamma, membership, improvement = future.result()
        memberships_by_gamma[gamma].append(membership)
        improvements_by_gamma[gamma].append(improvement)
        print(f"Completed Leiden run {completed:02d}/{len(tasks)}")

for gamma in GAMMA_CANDIDATES:
    memberships_by_gamma[gamma] = np.stack(memberships_by_gamma[gamma])
    expected_shape = (N_RUNS, len(graphs), len(roi_names))
    assert memberships_by_gamma[gamma].shape == expected_shape, (gamma, memberships_by_gamma[gamma].shape)

print(f"\nRan {len(tasks)} Leiden optimisations across {len(GAMMA_CANDIDATES)} gamma candidates ({N_RUNS} runs each)")

Completed Leiden run 01/40
Completed Leiden run 02/40
Completed Leiden run 03/40
Completed Leiden run 04/40
Completed Leiden run 05/40
Completed Leiden run 06/40
Completed Leiden run 07/40
Completed Leiden run 08/40
Completed Leiden run 09/40
Completed Leiden run 10/40
Completed Leiden run 11/40
Completed Leiden run 12/40
Completed Leiden run 13/40
Completed Leiden run 14/40
Completed Leiden run 15/40
Completed Leiden run 16/40
Completed Leiden run 17/40
Completed Leiden run 18/40
Completed Leiden run 19/40
Completed Leiden run 20/40
Completed Leiden run 21/40
Completed Leiden run 22/40
Completed Leiden run 23/40
Completed Leiden run 24/40
Completed Leiden run 25/40
Completed Leiden run 26/40
Completed Leiden run 27/40
Completed Leiden run 28/40
Completed Leiden run 29/40
Completed Leiden run 30/40
Completed Leiden run 31/40
Completed Leiden run 32/40
Completed Leiden run 33/40
Completed Leiden run 34/40
Completed Leiden run 35/40
Completed Leiden run 36/40
Completed Leiden run 37/40
C

## Community-count distribution per gamma

For each gamma, count the number of distinct communities in every window of every run, then summarize that distribution. This tells us which gamma values keep the scan in the 5–7 community range we're targeting.

In [9]:
rows = []
for gamma in GAMMA_CANDIDATES:
    counts = np.asarray([
        [len(np.unique(window_membership)) for window_membership in run]
        for run in memberships_by_gamma[gamma]
    ]).ravel()
    rows.append({
        "gamma": gamma,
        "mean_communities": counts.mean(),
        "median_communities": np.median(counts),
        "std": counts.std(),
        "p25": np.percentile(counts, 25),
        "p75": np.percentile(counts, 75),
        "min": counts.min(),
        "max": counts.max(),
        "mean_quality_improvement": np.mean(improvements_by_gamma[gamma]),
    })

gamma_calibration = pd.DataFrame(rows)
gamma_calibration["in_target_range_5_7"] = gamma_calibration["median_communities"].between(5, 7)
display(gamma_calibration)

gamma_calibration.to_csv(OUTPUT_DIR / "gamma_calibration.csv", index=False)
print(f"\nGamma values with median community count in [5, 7]: "
      f"{gamma_calibration.loc[gamma_calibration['in_target_range_5_7'], 'gamma'].tolist()}")

,gamma,mean_communities,median_communities,std,p25,p75,min,max,mean_quality_improvement,in_target_range_5_7
0,0.15,3.293231,3.0,0.747828,3.0,4.0,2,6,17909.755826,False
1,0.20,4.105846,4.0,0.864619,4.0,5.0,2,7,15200.196312,False
2,0.25,4.831077,5.0,0.992628,4.0,5.0,2,8,13067.129341,True
3,0.30,5.520615,5.0,1.080045,5.0,6.0,2,9,11310.864524,True
4,0.35,6.315692,6.0,1.121551,6.0,7.0,2,10,9875.115827,True
5,0.40,7.104308,7.0,1.193268,6.0,8.0,3,11,8682.533701,True
6,0.45,8.075385,8.0,1.311081,7.0,9.0,4,12,7698.711083,False
7,0.50,9.034154,9.0,1.352969,8.0,10.0,5,13,6912.180280,False



Gamma values with median community count in [5, 7]: [0.25, 0.3, 0.35, 0.4]


## How to interpret the output

- `gamma_calibration.csv` has one row per gamma candidate with the per-window community-count distribution (mean/median/std/IQR/min/max) at `omega = 0.25`.
- Use `median_communities` (more robust to outlier windows than the mean) to judge whether a gamma value sits in the 5–7 target range — the `in_target_range_5_7` column flags this directly.
- This is a single-scan, 5-run probe meant to narrow down candidates quickly, not a precision estimate — expect some run-to-run and window-to-window noise (the original 20-run/0.35 baseline showed counts ranging 2–10 per window with a 6.32 mean, so don't over-index on any single window).
- Pick 3–4 gamma values from the rows where `in_target_range_5_7` is `True` (or closest to it) to carry into the full (gamma, omega) parameter sweep.

## Full gamma × omega grid from `parameters_inspection.py`

The cells below read `GAMMAS` and `OMEGAS` directly from `parameters_inspection.py`, rather than copying the values into the notebook. The same `Bf_DTA_anes` mouse and all of its selected scan windows are used.

For every gamma/omega combination, temporal Leiden is run five times with the same `CPMVertexPartition` call used by parameter inspection. Within each run, the number of communities is counted separately in every window and averaged. The final `average_communities` value is the mean of those five run-level averages.

In [10]:
# Import the actual parameter-inspection module so this notebook always uses
# its current gamma/omega arrays and its exact graph-construction functions.
parameter_spec = importlib.util.spec_from_file_location(
    "parameters_inspection_grid",
    scripts_dir / "parameters_inspection.py",
)
parameter_inspection = importlib.util.module_from_spec(parameter_spec)
sys.modules[parameter_spec.name] = parameter_inspection
parameter_spec.loader.exec_module(parameter_inspection)

GRID_GAMMAS = list(parameter_inspection.GAMMAS)
GRID_OMEGAS = list(parameter_inspection.OMEGAS)
GRID_N_RUNS = 5
GRID_RANDOM_SEED = 20260624

# Rebuild this mouse's temporal graphs with parameters_inspection.read_window
# and parameters_inspection.build_igraph, including its signed edge weights.
parameter_graphs = []
parameter_roi_names = None
for path in window_paths:
    frame = parameter_inspection.read_window(str(path))
    current_rois = list(frame.columns)
    if parameter_roi_names is None:
        parameter_roi_names = current_rois
    else:
        assert current_rois == parameter_roi_names, f"ROI order changed in {path}"
    parameter_graphs.append(parameter_inspection.build_igraph(frame))

print(f"Mouse/scan: {SCAN_ID}")
print(f"Windows: {len(parameter_graphs)}")
print(f"Gamma values: {GRID_GAMMAS}")
print(f"Omega values: {GRID_OMEGAS}")
print(f"Combinations: {len(GRID_GAMMAS) * len(GRID_OMEGAS)}; runs per combination: {GRID_N_RUNS}")

Mouse/scan: sub-ag230908a_EXP_bold_parcellated
Windows: 650
Gamma values: [0.25, 0.3, 0.35, 0.4]
Omega values: [0.1, 0.2, 0.3, 0.4]
Combinations: 16; runs per combination: 5


In [11]:
_PARAMETER_GRID_GRAPHS = None


def _initialize_parameter_grid_worker(worker_graphs):
    global _PARAMETER_GRID_GRAPHS
    _PARAMETER_GRID_GRAPHS = worker_graphs


def _run_parameter_grid_leiden(gamma, omega, seed, run_number):
    """One temporal-Leiden optimisation using the parameter-inspection call."""
    memberships, improvement = la.find_partition_temporal(
        _PARAMETER_GRID_GRAPHS,
        la.CPMVertexPartition,
        interslice_weight=omega,
        vertex_id_attr="id",
        weight_attr="weight",
        resolution_parameter=gamma,
        seed=int(seed),
    )
    if not np.isfinite(improvement):
        raise RuntimeError(f"Temporal Leiden returned invalid improvement: {improvement}")

    membership_matrix = np.asarray(memberships, dtype=np.int32)
    expected_shape = (len(_PARAMETER_GRID_GRAPHS), _PARAMETER_GRID_GRAPHS[0].vcount())
    if membership_matrix.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected membership shape {membership_matrix.shape}; expected {expected_shape}"
        )

    communities_per_window = np.asarray([
        len(np.unique(window_membership))
        for window_membership in membership_matrix
    ])
    return {
        "gamma": gamma,
        "omega": omega,
        "run": run_number,
        "seed": int(seed),
        "mean_communities": communities_per_window.mean(),
        "median_communities": np.median(communities_per_window),
        "min_communities": communities_per_window.min(),
        "max_communities": communities_per_window.max(),
        "quality_improvement": float(improvement),
    }

In [12]:
# Reuse the same five seeds for every combination, making comparisons across
# gamma and omega less sensitive to a different random draw.
grid_seed_generator = np.random.default_rng(GRID_RANDOM_SEED)
grid_seeds = grid_seed_generator.integers(
    0, 2**31 - 1, size=GRID_N_RUNS, dtype=np.int64
)
grid_tasks = [
    (gamma, omega, int(seed), run_number)
    for gamma in GRID_GAMMAS
    for omega in GRID_OMEGAS
    for run_number, seed in enumerate(grid_seeds, start=1)
]

grid_run_rows = []
grid_workers = min(
    parameter_inspection.RUN_WORKERS,
    multiprocessing.cpu_count(),
    len(grid_tasks),
)
grid_mp_context = multiprocessing.get_context("fork")

with ProcessPoolExecutor(
    max_workers=grid_workers,
    mp_context=grid_mp_context,
    initializer=_initialize_parameter_grid_worker,
    initargs=(parameter_graphs,),
) as executor:
    futures = [executor.submit(_run_parameter_grid_leiden, *task) for task in grid_tasks]
    for completed, future in enumerate(as_completed(futures), start=1):
        grid_run_rows.append(future.result())
        print(f"Completed Leiden run {completed:02d}/{len(grid_tasks)}")

parameter_grid_runs = (
    pd.DataFrame(grid_run_rows)
    .sort_values(["gamma", "omega", "run"])
    .reset_index(drop=True)
)
print(f"Completed {len(parameter_grid_runs)} optimisations using {grid_workers} workers")

Completed Leiden run 01/80
Completed Leiden run 02/80
Completed Leiden run 03/80
Completed Leiden run 04/80
Completed Leiden run 05/80
Completed Leiden run 06/80
Completed Leiden run 07/80
Completed Leiden run 08/80
Completed Leiden run 09/80
Completed Leiden run 10/80
Completed Leiden run 11/80
Completed Leiden run 12/80
Completed Leiden run 13/80
Completed Leiden run 14/80
Completed Leiden run 15/80
Completed Leiden run 16/80
Completed Leiden run 17/80
Completed Leiden run 18/80
Completed Leiden run 19/80
Completed Leiden run 20/80
Completed Leiden run 21/80
Completed Leiden run 22/80
Completed Leiden run 23/80
Completed Leiden run 24/80
Completed Leiden run 25/80
Completed Leiden run 26/80
Completed Leiden run 27/80
Completed Leiden run 28/80
Completed Leiden run 29/80
Completed Leiden run 30/80
Completed Leiden run 31/80
Completed Leiden run 32/80
Completed Leiden run 33/80
Completed Leiden run 34/80
Completed Leiden run 35/80
Completed Leiden run 36/80
Completed Leiden run 37/80
C

In [13]:
parameter_grid_summary = (
    parameter_grid_runs
    .groupby(["gamma", "omega"], as_index=False)
    .agg(
        average_communities=("mean_communities", "mean"),
        between_run_std=("mean_communities", "std"),
        average_window_median=("median_communities", "mean"),
        min_communities=("min_communities", "min"),
        max_communities=("max_communities", "max"),
        average_quality_improvement=("quality_improvement", "mean"),
        n_runs=("run", "count"),
    )
    .sort_values(["gamma", "omega"])
    .reset_index(drop=True)
)

average_community_grid = parameter_grid_summary.pivot(
    index="gamma",
    columns="omega",
    values="average_communities",
)

print("Average number of communities (rows = gamma, columns = omega):")
display(average_community_grid.round(3))
display(parameter_grid_summary.round(3))

GRID_OUTPUT_DIR = ROOT / "outputs" / "community_interpretation" / "gamma_omega_grid_5runs"
GRID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
parameter_grid_runs.to_csv(GRID_OUTPUT_DIR / "gamma_omega_runs.csv", index=False)
parameter_grid_summary.to_csv(GRID_OUTPUT_DIR / "gamma_omega_summary.csv", index=False)
print(f"Saved results to {GRID_OUTPUT_DIR}")

Average number of communities (rows = gamma, columns = omega):


omega,0.1,0.2,0.3,0.4
gamma,,,,
0.25,4.951,4.995,5.009,5.024
0.30,5.535,5.597,5.650,5.698
0.35,6.198,6.306,6.357,6.418
0.40,6.942,7.063,7.164,7.265


,gamma,omega,average_communities,between_run_std,average_window_median,min_communities,max_communities,average_quality_improvement,n_runs
0,0.25,0.1,4.951,0.013,5.0,2,9,10211.261,5
1,0.25,0.2,4.995,0.032,5.0,2,8,12026.419,5
2,0.25,0.3,5.009,0.014,5.0,2,9,13866.764,5
3,0.25,0.4,5.024,0.047,5.0,2,9,15736.624,5
4,0.30,0.1,5.535,0.027,5.2,2,9,8535.497,5
5,0.30,0.2,5.597,0.025,5.7,2,9,10345.297,5
6,0.30,0.3,5.650,0.034,6.0,3,9,12195.921,5
7,0.30,0.4,5.698,0.019,6.0,3,9,14088.740,5
8,0.35,0.1,6.198,0.010,6.0,3,9,7111.134,5
9,0.35,0.2,6.306,0.026,6.0,3,9,8945.134,5


Saved results to /home/luca.galli-1/neuro3ducate/mouse-fmri-dyncomm/outputs/community_interpretation/gamma_omega_grid_5runs
